# VAW Phase 1 — Colab run: setup + M1 (manifest) + M2 (preprocess)
Cells only: mount Drive, get the code, install, test, run scripts, inspect. No pipeline logic lives here.
Read `colab/README.md` first (one-time setup: Drive shortcut + GitHub token secret).

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get the code (clone the first time, pull after)
Needs a Colab Secret named `GITHUB_TOKEN` (see README). The token is never printed or saved in this notebook.

In [ ]:
import os
from google.colab import userdata
REPO = 'KRITHIKSAI16/Violence-Against-Women'
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{REPO}.git'
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!git log --oneline | head -3

## 3. Install requirements and check the environment
`pytest` is a quick check that this Colab session is set up right (ffmpeg, OpenCV).

In [ ]:
!pip install -q -r requirements.txt
!ffmpeg -version | head -1
!python -m pytest -q

## 4. Check the Drive paths in `configs/colab.yaml`

In [ ]:
import os, yaml
cfg = yaml.safe_load(open('configs/colab.yaml'))
print('data_root exists:', os.path.isdir(cfg['data_root']), '->', cfg['data_root'])
print('categories     :', sorted(d for d in os.listdir(cfg['data_root']) if os.path.isdir(os.path.join(cfg['data_root'], d))) if os.path.isdir(cfg['data_root']) else 'N/A')
os.makedirs(os.path.dirname(cfg['manifest_path']), exist_ok=True)

## 5. M1 — Data Acquisition (manifest)

In [ ]:
!python -m src.data.manifest --config configs/colab.yaml

## 6. M2 — Preprocessing (resumable: re-running skips clips already done)

In [ ]:
!python -m src.data.preprocess --config configs/colab.yaml

## 7. Sanity checks on the results

In [ ]:
import json, collections
m = json.load(open(cfg['manifest_path']))
c = json.load(open(cfg['preprocess']['clean_manifest_path']))
print('M1 clips:', m['num_clips'], '| skipped:', len(m['skipped']))
print('M2 clips:', c['num_clips'], '| skipped:', len(c['skipped']))
print('per category (M2):', c['counts_per_category'])
ids = [x['clip_id'] for x in c['clips']]
dups = [k for k, n in collections.Counter(ids).items() if n > 1]
print('duplicate clip_ids:', dups or 'none')
for s in m['skipped'] + c['skipped']:
    print('SKIPPED', s['path'], '-', s['reason'])